<a href="https://www.kaggle.com/code/kaoutharhamdan/nlp-pipeline-week-2?scriptVersionId=318714928" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Importation des bibliothèques

In [ ]:
!pip install PyPDF2



In [ ]:
import re
import pandas as pd
import numpy as np

# Arabic preprocessing
# Tester d'autre biblio !!
try:
    from pyarabic.araby import strip_tashkeel
except:
    print("Installer pyarabic : pip install pyarabic")

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [ ]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        num_pages = min(4, len(reader.pages))
        
        for i in range(1,num_pages):
            page = reader.pages[i]
            text += page.extract_text()
            
    return text

In [ ]:
raw_text=extract_text_from_pdf("/kaggle/input/datasets/kaoutharhamdan/code-de-route-ma52/code de la route MA52_05.pdf")

In [ ]:
print(raw_text)

## Prétraitement arabe

In [ ]:
def fix_common_arabic_pdf_errors(text):
    replacements = {
        "املادة": "المادة",
        "املركبة": "المركبة",
        "املركبات": "المركبات",
        "السري": "السير",
        "الطرق": "الطرق",}

    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)

    return text

In [ ]:
raw_text=fix_common_arabic_pdf_errors(raw_text)

In [ ]:
print(raw_text)

In [ ]:
import re
from pyarabic.araby import strip_tashkeel, strip_tatweel

def normalize_arabic(text):
    # 1. Remove tashkeel (harakat/vowels)
    text = strip_tashkeel(text)
    
    # 2. Remove tatweel (elongation character ـ)
    text = strip_tatweel(text)
    
    # 3. Normalize Hamza variants → ا
    text = re.sub(r"[أإآ]", "ا", text)
    
    # 4. Hamza on waw/ya → bare letters
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    
    # 5. Ta Marbuta → ه
    text = re.sub(r"ة", "ه", text)
    
    # 6. Ya variants → ي
    text = re.sub(r"ى", "ي", text)
    
    # 7. Remove Arabic punctuation that breaks regex
    text = re.sub(r"[،؛؟٪]", " ", text)


    
    
    return text

clean_text = normalize_arabic(raw_text)
print(clean_text)

## Segmentation en articles

In [ ]:
def segment_articles(text):
    # FIX 1: Remove \b — it doesn't work with Arabic. Use a lookahead/lookbehind on digits instead.
    # FIX 2: Make "الماده" optional prefix with ال to cover both written forms.
    # FIX 3: Use (?:^|\n) or just rely on the digit boundary to anchor the match.
    pattern = r'((?:(?<=\n)|(?<=^))(?:\d+\s+(?:ال)?ماده|(?:ال)?ماده\s+\d+))'
    parts = re.split(pattern, text, flags=re.MULTILINE)
    
    articles = []
    for i in range(1, len(parts), 2):
        header = parts[i].strip()
        body = parts[i+1].strip() if i+1 < len(parts) else ""
        if len(body) < 30:
            continue
        articles.append({
            "header": header,
            "body": body
        })
    return articles

articles = segment_articles(clean_text)



articles = segment_articles(clean_text)

for a in articles:
    print("------")
    print(a)

## Extraction par règles (Regex)

In [ ]:
def extract_entities(article):
    header = article["header"]
    body   = article["body"]
    result = {}

    # ── Article ID ──────────────────────────────────────────
    art = re.search(r"(\d+)", header)
    result["article_id"] = art.group(1) if art else None

    # ── Description ─────────────────────────────────────────
    result["infraction_desc"] = re.split(r"[.؟!]", body)[0].strip()[:300]

    # ── Amende fixe & range ──────────────────────────────────
    fine = re.search(r"(\d+)\s+درهم", body)
    result["amende_fixe"] = fine.group(1) if fine else None

    fine_range = re.search(r"(\d+)\s+(?:إلي|الي|إلى)\s+(\d+)\s+درهم", body)
    result["amende_min"] = fine_range.group(1) if fine_range else None
    result["amende_max"] = fine_range.group(2) if fine_range else None

    # ── Points retrait ───────────────────────────────────────
    points = re.search(r"(?:خصم|سحب)\s+(\d+)\s+(?:نقط|نقاط)", body)
    result["points_retrait"] = int(points.group(1)) if points else None

    # ── Emprisonnement ───────────────────────────────────────
    prison = re.search(r"(?:الحبس|حبس)\s+(?:من\s+)?(\d+)\s+(?:يوم|شهر|اشهر)", body)
    result["emprisonnement"] = prison.group(0) if prison else None

    # ── Categorie véhicule ───────────────────────────────────
    vehicule_map = {
        "دراجه ناريه":    "moto",
        "دراجات الناريه": "moto",
        "دراجه رباعيه":   "quad",
        "مركبه ثقيله":    "poids_lourd",
        "مركبه فالحيه":   "vehicule_agricole",
        "مركبه غابويه":   "vehicule_forestier",
        "سياره":           "vehicule_leger",
        "حافله":           "autobus",
        "شاحنه":           "camion",
        "مركبه عسكريه":   "militaire",
        "عربه":            "remorque",
    }
    found_cats = [label for kw, label in vehicule_map.items() if kw in body]
    result["categorie_vehicule"] = ", ".join(found_cats) if found_cats else None

    # ── Catégorie permis ─────────────────────────────────────
    licence = re.findall(r"\b(AM|A1|A|B|C|D|E\(B\)|E\(C\)|E\(D\))\b", body)
    result["categorie_permis"] = ", ".join(set(licence)) if licence else None

    # ── Chapitre (الباب) ─────────────────────────────────────
    bab = re.search(r"الباب\s+(الاول|الثاني|الثالث|الرابع|الخامس|\w+)", body)
    result["chapitre"] = bab.group(0) if bab else None

    # ── Role paragraphe (rule-based) ─────────────────────────
    sanction_kw     = ["يعاقب", "غرامه", "درهم", "حبس", "خصم"]
    interdiction_kw = ["لا يجوز", "يحظر", "ممنوع"]
    obligation_kw   = ["يجب", "يلتزم", "يتعين"]
 

    if any(k in body for k in sanction_kw):
        result["role_paragraphe"] = "sanction"
    elif any(k in body for k in interdiction_kw):
        result["role_paragraphe"] = "interdiction"
    elif any(k in body for k in obligation_kw):
        result["role_paragraphe"] = "obligation"
    else:
        result["role_paragraphe"] = "definition"

    # ── NEW: Type infraction ──────────────────────────────────
    # More specific than role — what KIND of violation is it?
    type_map = {
        "سرعه":                 "excès_de_vitesse",
        "كحول":                 "conduite_en_état_d_ivresse",
        "هاتف":                 "usage_téléphone",
        "تجاوز":                "dépassement",
        "وقوف":                 "stationnement",
        "توقف":                 "stationnement",
        "حزام":                 "ceinture",
        "خوذه":                 "casque",
        "ضوء احمر":             "feu_rouge",
        "رخصه السياقه":         "défaut_de_permis",
        "تامين":                "défaut_assurance",
        "فحص تقني":             "défaut_contrôle_technique",
        "اوراق":                "défaut_de_documents",
        "سكر":                  "conduite_en_état_d_ivresse",
        "مخدر":                 "conduite_sous_stupéfiants",
    }
    found_types = [label for kw, label in type_map.items() if kw in body]
    result["type_infraction"] = ", ".join(found_types) if found_types else "non_classifié"

    # ── NEW: Niveau gravité ───────────────────────────────────
    # Based on fine amount and/or points lost
    amende = int(result["amende_fixe"]) if result["amende_fixe"] else 0
    pts    = result["points_retrait"]   or 0

    if result["emprisonnement"] or amende >= 3000 or pts >= 4:
        result["niveau_gravite"] = "élevé"
    elif amende >= 1000 or pts >= 2:
        result["niveau_gravite"] = "moyen"
    elif amende > 0 or pts > 0:
        result["niveau_gravite"] = "faible"
    else:
        result["niveau_gravite"] = "non_défini"

    # ── NEW: Localisation ─────────────────────────────────────
    localisation_map = {
        "طريق سيار":       "autoroute",
        "اوتوروت":         "autoroute",
        "طريق عمومي":      "route_nationale",
        "الطريق العموميه": "route_nationale",
        "داخل المدينه":    "zone_urbaine",
        "تجمع عمراني":     "zone_urbaine",
        "خارج المدينه":    "hors_agglomération",
        "منطقه مدرسيه":    "zone_scolaire",
        "ممر للمشاه":      "passage_piéton",
    }
    found_locs = [label for kw, label in localisation_map.items() if kw in body]
    result["localisation"] = ", ".join(found_locs) if found_locs else None

    # ── NEW: Suspension permis ────────────────────────────────
    suspension = re.search(r"(?:سحب|توقيف|إيقاف)\s+(?:رخصه|الرخصه)\s+(?:السياقه)?", body)
    result["suspension_permis"] = "oui" if suspension else "non"

    # ── NEW: Concerne piéton ──────────────────────────────────
    pieton_kw = ["مشاه", "راجل", "عابر"]
    result["concerne_pieton"] = "oui" if any(k in body for k in pieton_kw) else "non"

    # ── NEW: Récidive mentionnée ──────────────────────────────
    recidive_kw = ["عوده", "تكرار", "مره ثانيه"]
    result["recidive"] = "oui" if any(k in body for k in recidive_kw) else "non"

    # ── Mots-clés ─────────────────────────────────────────────
    keywords_map = {
        "vitesse":       ["سرعه", "سرعه مفرطه"],
        "stationnement": ["وقوف", "توقف", "ركن"],
        "alcool":        ["كحول", "سكر"],
        "telephone":     ["هاتف", "جهاز نقال"],
        "nuit":          ["ليل", "الليل"],
        "autoroute":     ["طريق سيار"],
        "ceinture":      ["حزام الامان"],
        "casque":        ["خوذه"],
        "feu_rouge":     ["ضوء احمر", "اشاره حمراء"],
        "depassement":   ["تجاوز"],
        "permis":        ["رخصه السياقه"],
    }
    found_kw = [tag for tag, pats in keywords_map.items() if any(p in body for p in pats)]
    result["mots_cles"] = ", ".join(found_kw) if found_kw else None

    return result


# ── Run & export ────────────────────────────────────────────
rows = [extract_entities(a) for a in articles]
df = pd.DataFrame(rows)
df.head()

In [ ]:
df['infraction_desc']

## Approche ML (TF-IDF)


transformer le texte en vecteurs numériques (se préparer au clustering ou étape suivante)


--------

(max_features=50) Cela signifie :

- garder au maximum les 50 mots les plus importants / fréquents du corpus

- Même si ton corpus contient 1000 mots distincts, seuls les 50 meilleurs termes seront conservés.

C’est utile pour :

- réduire la dimension
- accélérer le calcul
- éviter le bruit

In [ ]:
vectorizer = TfidfVectorizer(max_features=50)

X = vectorizer.fit_transform(df["infraction_desc"])

print(X.shape)

## Clustering des infractions

- KMeans est l’algorithme de clustering non supervisé qui regroupe les données en K groupes (clusters).

L’idée est de :

- choisir K centres (appelés centroïdes)
- affecter chaque point au centre le plus proche
- recalculer les centres
- répéter jusqu’à stabilisation

- Random state : 
K-Means commence par choisir des centres initiaux au hasard. Sans fixer cette valeur, les résultats peuvent changer à chaque exécution

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42)

df["cluster"] = kmeans.fit_predict(X)

df[["article_id", "cluster"]]

In [ ]:
# Récupérer les clusters
labels = kmeans.labels_
print(labels)

In [ ]:
#using it on the whole pdf

import PyPDF2

def extract_text_from_pdf2(pdf_path):
    text = ""
    
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:  # avoid None
                text += page_text + "\n"
    
    return text


In [ ]:
raw1=extract_text_from_pdf2('/kaggle/input/datasets/kaoutharhamdan/code-de-route-ma52/code de la route MA52_05.pdf')
raw1=fix_common_arabic_pdf_errors(raw1)
clean_text = normalize_arabic(raw1)


In [ ]:
articles = segment_articles(clean_text)
rows1 = [extract_entities(a) for a in articles]
df = pd.DataFrame(rows1)
df.to_csv("export_final2.csv", index=False, encoding="utf-8-sig")
df.head(16)

## Approche modèle pré-entraîné 

In [ ]:
!pip install camel-tools --break-system-packages
!pip install transformers torch sentence-transformers --break-system-packages

# Download CAMeL Tools data
!python -m camel_tools.cli.camel_data -i morphology-db-msa-r13

In [ ]:
from camel_tools.tokenizers.word import simple_word_tokenize

def tokenize_article(body):
    tokens = simple_word_tokenize(body)
    return tokens

print(tokenize_article(articles[0]["body"]))

In [ ]:
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

db       = MorphologyDB.builtin_db()
analyzer = Analyzer(db)

def analyze_tokens(body):
    tokens  = tokenize_article(body)
    results = []
    for tok in tokens[:10]:            
        analyses = analyzer.analyze(tok)
        if analyses:
            top = analyses[0]
            results.append({
                "token": tok,
                "lemma": top.get("lex"),
                "pos":   top.get("pos"),
            })
    return results

analyze_tokens(articles[0]["body"])

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
embedder = SentenceTransformer("CAMeL-Lab/bert-base-arabic-camelbert-msa")

def embed_article(body):
    return embedder.encode(body, show_progress_bar=False)


vec = embed_article(articles[0]["body"])
print(vec.shape)

In [ ]:
# embed every article body (not just one)
bodies     = [a["body"] for a in articles]
embeddings = embedder.encode(bodies, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")  # should be (num_articles, 768)

In [ ]:
from sklearn.cluster import KMeans

n_clusters = 2

km          = KMeans(n_clusters=n_clusters, random_state=42)
bert_labels = km.fit_predict(embeddings)

# attach cluster to each article
for i, a in enumerate(articles):
    a["cluster_bert"] = int(bert_labels[i])

# quick check — what articles landed in each cluster?
for cluster_id in range(n_clusters):
    print(f"\n── Cluster {cluster_id} ──")
    for a in articles:
        if a["cluster_bert"] == cluster_id:
            print(f"  {a['header']} | {a['body'][:60]}...")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca    = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

plt.figure(figsize=(8, 5))
for cluster_id in range(n_clusters):
    idx = [i for i, l in enumerate(bert_labels) if l == cluster_id]
    plt.scatter(coords[idx, 0], coords[idx, 1],
                label=f"Cluster {cluster_id}", s=80)

# label each point with its article number
for i, a in enumerate(articles):
    plt.annotate(a["header"], (coords[i, 0], coords[i, 1]),
                 fontsize=7, alpha=0.7)

plt.legend()
plt.title("AraBERT article clusters")
plt.tight_layout()
plt.savefig("clusters_bert.png", dpi=150)
plt.show()